In [2]:
# ======================
# IMPORTS
# ======================
import os
import cv2
import numpy as np
from tqdm import tqdm


# ======================
# CONFIG
# ======================
INPUT_PATH = "/kaggle/input/datasets/shanthoshkumarr/gf-final/golden_frames_final"
OUTPUT_PATH = "/kaggle/working/face_crops"

IMG_SIZE = 224
MARGIN = 0.2

CLASS_NAMES = [
    "DeepFakeDetection",
    "Deepfakes",
    "Face2Face",
    "FaceShifter",
    "FaceSwap",
    "NeuralTextures",
    "original"
]

os.makedirs(OUTPUT_PATH, exist_ok=True)


# ======================
# LOAD FACE DETECTOR
# ======================
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)


# ======================
# STATS
# ======================
total_images = 0
success_faces = 0
failed_faces = 0


# ======================
# CENTER CROP
# ======================
def center_crop(img):
    h, w, _ = img.shape
    size = min(h, w)

    cx, cy = w // 2, h // 2

    x1 = max(cx - size // 2, 0)
    y1 = max(cy - size // 2, 0)
    x2 = x1 + size
    y2 = y1 + size

    return img[y1:y2, x1:x2]


# ======================
# MAIN LOOP
# ======================
for class_name in CLASS_NAMES:

    in_dir = os.path.join(INPUT_PATH, class_name)
    out_dir = os.path.join(OUTPUT_PATH, class_name)

    os.makedirs(out_dir, exist_ok=True)

    print(f"\nProcessing class: {class_name}")

    for root, _, files in os.walk(in_dir):

        for file in tqdm(files):

            if not file.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            total_images += 1

            img_path = os.path.join(root, file)
            img = cv2.imread(img_path)

            if img is None:
                continue

            # ======================
            # PRESERVE VIDEO FOLDER
            # ======================
            rel_path = os.path.relpath(root, in_dir)
            save_dir = os.path.join(out_dir, rel_path)
            os.makedirs(save_dir, exist_ok=True)

            # ======================
            # FACE DETECTION
            # ======================
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            faces = face_cascade.detectMultiScale(
                gray,
                scaleFactor=1.3,
                minNeighbors=5
            )

            h, w, _ = img.shape

            if len(faces) > 0:

                # pick largest face
                x, y, bw, bh = max(faces, key=lambda b: b[2] * b[3])

                # add margin
                x1 = int(x - MARGIN * bw)
                y1 = int(y - MARGIN * bh)
                x2 = int(x + bw + MARGIN * bw)
                y2 = int(y + bh + MARGIN * bh)

                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(w, x2)
                y2 = min(h, y2)

                crop = img[y1:y2, x1:x2]
                success_faces += 1

            else:
                crop = center_crop(img)
                failed_faces += 1

            # ======================
            # RESIZE
            # ======================
            crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))

            # ======================
            # SAVE IMAGE
            # ======================
            save_path = os.path.join(save_dir, file)
            cv2.imwrite(save_path, crop)


# ======================
# FINAL STATS
# ======================
print("\n" + "="*60)
print("FACE EXTRACTION COMPLETE")
print("="*60)
print(f"Total Images       : {total_images}")
print(f"Face Detected      : {success_faces}")
print(f"Fallback Used      : {failed_faces}")
print(f"Detection Success  : {success_faces / total_images * 100:.2f}%")


Processing class: DeepFakeDetection


0it [00:00, ?it/s]
100%|██████████| 3/3 [00:00<00:00,  4.13it/s]



Processing class: Deepfakes


0it [00:00, ?it/s]
100%|██████████| 8/8 [00:01<00:00,  7.72it/s]



Processing class: Face2Face


...


Processing class: NeuralTextures


0it [00:00, ?it/s]
100%|██████████| 3/3 [00:00<00:00,  7.56it/s]



Processing class: original


0it [00:00, ?it/s]
100%|██████████| 8/8 [00:00<00:00, 19.30it/s]


FACE EXTRACTION COMPLETE
Total Images       : 40984
Face Detected      : 39426
Fallback Used      : 1558
Detection Success  : 96.20%


In [4]:
import shutil

shutil.make_archive(
    '/kaggle/working/output',  # name of zip file (no .zip)
    'zip',
    '/kaggle/working'          # folder to compress
)

'/kaggle/working/output.zip'